# Customer Churn Prediction: Profit-Based Evaluation of ML Models
## Step 5: Model Training

In [ ]:
# ── IMPORTS ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time

from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

sns.set_theme(style='whitegrid')

# Load preprocessed splits
X_train = pd.read_csv('X_train.csv')
X_test  = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
y_test  = pd.read_csv('y_test.csv').squeeze()

print(f'Data loaded ✓')
print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')

### 5.1 — Define the Three Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        random_state=42,
        solver='lbfgs'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
}

print('Models defined:')
for name in models:
    print(f'  • {name}')

### 5.2 — 5-Fold Cross Validation on Training Set

In [ ]:
# 5-fold stratified CV — as described in your research proposal
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}

print('Running 5-Fold Cross Validation...\n')
for name, model in models.items():
    start = time.time()
    scores = cross_val_score(model, X_train, y_train,
                             cv=cv, scoring='roc_auc', n_jobs=-1)
    elapsed = time.time() - start
    cv_results[name] = scores
    print(f'{name}')
    print(f'  AUC per fold : {[round(s,4) for s in scores]}')
    print(f'  Mean AUC     : {scores.mean():.4f} ± {scores.std():.4f}')
    print(f'  Time         : {elapsed:.1f}s\n')

In [ ]:
# Plot CV results
fig, ax = plt.subplots(figsize=(9, 5))

colors = ['#2E86AB', '#E74C3C', '#2ECC71']
names  = list(cv_results.keys())
means  = [cv_results[n].mean() for n in names]
stds   = [cv_results[n].std()  for n in names]

bars = ax.bar(names, means, color=colors, edgecolor='white',
              width=0.5, yerr=stds, capsize=8,
              error_kw={'elinewidth': 2, 'ecolor': 'black'})

for bar, mean, std in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + std + 0.005,
            f'{mean:.4f}', ha='center', fontweight='bold', fontsize=11)

ax.set_title('5-Fold Cross Validation — Mean AUC (Training Set)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Mean ROC-AUC Score')
ax.set_ylim(0.75, 0.95)
ax.axhline(0.8, color='gray', linestyle='--', linewidth=1, label='AUC = 0.80 baseline')
ax.legend()
plt.tight_layout()
plt.savefig('plot_11_cv_auc.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved ✓')

### 5.3 — Train Final Models on Full Training Set

In [ ]:
trained_models = {}

print('Training final models on full training set...\n')
for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    trained_models[name] = model
    print(f'✓ {name} trained in {elapsed:.2f}s')

print('\nAll models trained on full training set ✓')

### 5.4 — Training Set Accuracy (Sanity Check)

In [ ]:
print('=== Training Accuracy (sanity check — not final evaluation) ===')
print('Note: High training accuracy for RF/GB is expected — test set scores come in Step 6\n')

for name, model in trained_models.items():
    train_acc = model.score(X_train, y_train)
    print(f'{name}: {train_acc*100:.2f}%')

### 5.5 — Feature Importance (Random Forest)

In [ ]:
rf_model = trained_models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
top15 = importances.nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(top15.index, top15.values, color='#2E86AB', edgecolor='white')

for bar, val in zip(bars, top15.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_title('Top 15 Feature Importances — Random Forest',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('plot_12_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 most important features:')
for feat, imp in importances.nlargest(5).items():
    print(f'  {feat}: {imp:.4f}')
print('\nInsight: tenure, MonthlyCharges, and TotalCharges dominate — aligns with EDA findings.')

### 5.6 — Save Trained Models

In [ ]:
for name, model in trained_models.items():
    filename = name.lower().replace(' ', '_') + '_model.pkl'
    joblib.dump(model, filename)
    print(f'Saved: {filename} ✓')

print('\n' + '='*55)
print('          MODEL TRAINING SUMMARY')
print('='*55)
for name in names:
    mean_auc = cv_results[name].mean()
    std_auc  = cv_results[name].std()
    print(f'  {name:<25} CV AUC: {mean_auc:.4f} ± {std_auc:.4f}')
print('='*55)
print('  All models saved as .pkl files')
print('  5-fold stratified CV completed')
print('='*55)
print('Step 5 Complete ✓  →  Proceed to Step 6: Traditional Evaluation')